##### After obtaining your API_KEY and API_SECRET from E-Trade, 
##### you can use the following template to execute your own real-time strategy

In [ ]:
#Set Key and Secret First
#It could be either sandbox or production

CONSUMER_KEY=""
CONSUMER_SECRET=""
SANDBOX_BASE_URL="https://apisb.etrade.com"
PROD_BASE_URL="https://api.etrade.com"

#Set the base url to either SANDBOX_BASE_URL or PROD_BASE_URL
base_url=SANDBOX_BASE_URL #Change to PROD_BASE_URL for production

In [ ]:
# Import necessary libraries

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import webbrowser
import logging
from logging.handlers import RotatingFileHandler
from rauth import OAuth1Service
import random
import time
import threading

In [ ]:
# Set up logging configuration
logger = logging.getLogger('my_logger')
logger.setLevel(logging.DEBUG)
handler = RotatingFileHandler("python_client.log", maxBytes=5*1024*1024, backupCount=3)
FORMAT = "%(asctime)-15s %(message)s"
fmt = logging.Formatter(FORMAT, datefmt='%m/%d/%Y %I:%M:%S %p')
handler.setFormatter(fmt)
logger.addHandler(handler)

headers = {"Content-Type": "application/xml", "consumerKey": CONSUMER_KEY}

etrade = OAuth1Service(
    name="etrade",
    consumer_key=CONSUMER_KEY,
    consumer_secret=CONSUMER_SECRET,
    request_token_url="https://api.etrade.com/oauth/request_token",
    access_token_url="https://api.etrade.com/oauth/access_token",
    authorize_url="https://us.etrade.com/e/t/etws/authorize?key={}&token={}",
    base_url="https://api.etrade.com"
)

# Get the request token
request_token, request_token_secret = etrade.get_request_token(
    params={"oauth_callback": "oob", "format": "json"}
)

# Open the browser to the authorize URL to get the verifier code
authorize_url = etrade.authorize_url.format(etrade.consumer_key, request_token)
webbrowser.open(authorize_url)

##### A browser window will automatically open to authorize access. After you log in, you will receive a verifier code. Assign this code to the text_code variable to authorize your session and gain access to your account."

In [ ]:
# Get the access token with the verifier code
text_code = ""
session = etrade.get_auth_session(request_token, request_token_secret, params={"oauth_verifier": text_code})

# Print session status
print(session)

In [ ]:
# Get the account list
account_list = session.get(base_url + "/v1/accounts/list.json")
account_list = pd.DataFrame(account_list.json()['AccountListResponse']['Accounts']['Account'])
account_list

#### Select the account you wish to trade with, obtain the accountIdKey, and assign it into below

In [ ]:
# Choose account to trade with and assign accountIdKey
accountIdKey = ""
previewurl = base_url + "/v1/accounts/" + accountIdKey + "/orders/preview.json"
orderurl = base_url + "/v1/accounts/" + accountIdKey + "/orders/place.json

#### Define function to fetch data, make predictions, execute trades, and close positions.

In [ ]:
def quant(stock, amount, threshold, interval):

    #get current stock quote and Price
    quoteurl=f"{base_url}/v1/market/quote/{stock}.json"
    response=session.get(quoteurl, header_auth=True)
    price=response.json()['QuoteResponse']['QuoteData'][0]['All']['lastTrade']


    #get today's date
    date = pd.Timestamp.today().strftime("%Y-%m-%d")

    # Download stock data from Yahoo Finance for Example you can use any other source you use to work with
    df = yf.download(stock, start=date, interval=interval)


    ######USE YOUR OWN STRATEGY HERE######
    ######It could be anything from a simple moving average crossover to a complex deep learning model######


    # Define the action based on the prediction
    action= "BUY" if action == 1 else "SELL_SHORT"

    
    ##########PREVIEW and PLACE ORDER##########

    # #Calculate the quantity of shares to buy        
    quantity=int(amount//price)

    #generate a random orderid
    orderid=str(random.randint(10000000000,99999999999))

    #preview order
    payload = f"""<PreviewOrderRequest>
                <orderType>EQ</orderType>
                <clientOrderId>{orderid}</clientOrderId>
                <Order>
                    <allOrNone>false</allOrNone>
                    <priceType>MARKET</priceType>  
                    <orderTerm>GOOD_FOR_DAY</orderTerm>   
                    <marketSession>REGULAR</marketSession>
                    <stopPrice></stopPrice>
                    <limitPrice></limitPrice>
                    <Instrument>
                        <Product>
                            <securityType>EQ</securityType>
                            <symbol>{stock}</symbol>
                        </Product>
                        <orderAction>{action}</orderAction> 
                        <quantityType>QUANTITY</quantityType>
                        <quantity>{quantity}</quantity>
                    </Instrument>
                </Order>
            </PreviewOrderRequest>"""

    #send the request
    response=session.post(previewurl,header_auth=True,headers=headers,data=payload)
    logger.debug("Payload: %s", payload)
    logger.debug("Request Header: %s", response.request.headers)
    logger.debug("Response: %s", response.json())

    #get the previewid
    previewId_data = response.json().get('PreviewOrderResponse', {}).get('PreviewIds', [{}])[0]
    previewId = previewId_data.get('previewId', None)
 
    #place order
    orderid2=str(random.randint(100000000,99999999999))
    payloads= f"""<PlaceOrderRequest>
                    <orderType>EQ</orderType>
                    <clientOrderId>{orderid2}</clientOrderId>
                    <PreviewIds>
                        <previewId>{previewId}</previewId>
                    </PreviewIds>    
                        <Order>
                            <allOrNone>false</allOrNone>
                            <priceType>MARKET</priceType>
                            <orderTerm>GOOD_FOR_DAY</orderTerm>
                            <marketSession>REGULAR</marketSession>
                            <stopPrice />
                            <limitPrice />
                            <Instrument>
                                <Product>
                                    <securityType>EQ</securityType>
                                    <symbol>{stock}</symbol>
                                </Product>
                                <orderAction>{action}</orderAction>
                                <quantityType>QUANTITY</quantityType>
                                <quantity>{quantity}</quantity>
                            </Instrument>
                        </Order>
                    </PlaceOrderRequest>"""
                
    response=session.post(orderurl,header_auth=True,headers=headers,data=payloads)
    logger.debug("Request Header: %s", response.request.headers)
    logger.debug("Request Payload: %s", payloads)
    logger.debug("Response: %s", response.json())

    #print the order detail response in dataframes justs like account list
    Order_Placed=response['PlaceOrderResponse']['Order'][0]['estimatedTotalAmount']
    #print order placed
    print(Order_Placed)



    ############TAKE-PROFIT and STOP-LOSS ORDERS##########



    # Calculate limit prices for take-profit and stop-loss orders
    if action == 'BUY':
        take_profit_limit_price = round(price * (1+threshold), 2) 
        stop_loss_limit_price = round(price * (1-threshold), 2)
        order_action = 'SELL'
    else:
        stop_loss_limit_price = round(price * (1-threshold), 2)
        take_profit_limit_price = round(price * (1+threshold), 2)
        order_action = 'BUY_TO_COVER'



    #Create loop to check spot price  for take-profit and stop-loss orders
        
    while True:
    
        response=session.get(quoteurl, header_auth=True)
        logger.debug(response.json())
        lastprice=response.json()['QuoteResponse']['QuoteData'][0]['All']['lastTrade']
        
        if lastprice >= take_profit_limit_price or lastprice <= stop_loss_limit_price:
                print(f"Target reached. Preparing to place closing order. Last trade price: {lastprice}")
                # Generate another unique client order ID for the take-profit order
                orderid_tp = str(random.randint(10000000000, 99999999999))
                # Preview Take-profit limit order or Stop Loss
                closepreview = f"""<PreviewOrderRequest>
                            <orderType>EQ</orderType>
                            <clientOrderId>{orderid_tp}</clientOrderId>
                            <Order>
                                <allOrNone>false</allOrNone>
                                <priceType>MARKET</priceType>  
                                <orderTerm>GOOD_FOR_DAY</orderTerm>   
                                <marketSession>REGULAR</marketSession>
                                <stopPrice />
                                <limitPrice />
                                <Instrument>
                                    <Product>
                                        <securityType>EQ</securityType>
                                        <symbol>{stock}</symbol>
                                    </Product>
                                    <orderAction>{order_action}</orderAction> 
                                    <quantityType>QUANTITY</quantityType>
                                    <quantity>{quantity}</quantity>
                                </Instrument>
                            </Order>
                        </PreviewOrderRequest>"""
                response=session.post(previewurl,header_auth=True,headers=headers,data=closepreview)
                logger.debug("Request Header: %s", response.request.headers)
                logger.debug("Request Payload: %s", closepreview)
                logger.debug("Response: %s", response.json())
                #get the previewid
                previewId_data = response.json().get('PreviewOrderResponse', {}).get('PreviewIds', [{}])[0]
                previewId = previewId_data.get('previewId', None)
                
                #Place take_profit order or stop loss order
                orderid2_tp=str(random.randint(100000000,99999999999))
                closeplace= f"""<PlaceOrderRequest>
                                <orderType>EQ</orderType>
                                <clientOrderId>{orderid2_tp}</clientOrderId>
                                <PreviewIds>
                                    <previewId>{previewId}</previewId>
                                </PreviewIds>    
                                    <Order>
                                        <allOrNone>false</allOrNone>
                                        <priceType>MARKET</priceType>
                                        <orderTerm>GOOD_FOR_DAY</orderTerm>
                                        <marketSession>REGULAR</marketSession>
                                        <stopPrice />
                                        <limitPrice />
                                        <Instrument>
                                            <Product>
                                                <securityType>EQ</securityType>
                                                <symbol>{stock}</symbol>
                                            </Product>
                                            <orderAction>{order_action}</orderAction>
                                            <quantityType>QUANTITY</quantityType>
                                            <quantity>{quantity}</quantity>
                                        </Instrument>
                                    </Order>
                                </PlaceOrderRequest>"""
                response=session.post(orderurl,header_auth=True,headers=headers,data=closeplace)
                logger.debug("Request Header: %s", response.request.headers)
                logger.debug("Request Payload: %s", closeplace)
                logger.debug("Response: %s", response.json())
                
                 #print the order detail response in dataframes justs like account list
                Order_Closed=response['PlaceOrderResponse']['Order'][0]['estimatedTotalAmount']
                #print order placed
                print(Order_Closed)
                print(Order_Closed)

                #Print Profit last trade
                print(f"Profit: {Order_Closed-Order_Placed}")


                break
        time.sleep(0.1)




In [ ]:
def run_continuously(stock, amount):
    while True:
        try:
            quant(stock, amount)
            print(f"Quant function execution completed for {stock}. Restarting...")
             
        except KeyboardInterrupt:
            print(f"Process manually interrupted by the user for {stock}. Exiting...")
            break
        except Exception as e:
            print(f"An error occurred during quant execution for {stock}: {e}")



if __name__ == "__main__":

    ###### List of stock or stocks to run the quant function on ######

    stocks = ["","",""]
    threads = []

    # Create and start a thread for each stock
    for stock in stocks:
        thread = threading.Thread(target=run_continuously, args=(stock, 21000))
        threads.append(thread)
        thread.start()

    # Wait for all threads to complete
    for thread in threads:
        thread.join()